In [ ]:
# ============================================================
# 环境配置
# - Colab 用户：取消注释下方 Colab 区块
# - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch matplotlib numpy -q

# ── 本地 Jupyter 环境 ──
import subprocess
import sys

def _install(pkg: str):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for package in ["torch", "matplotlib", "numpy"]:
    _install(package)


# GPT-1 代码实战：学习实现 vs 工程实现

基于论文 *Improving Language Understanding by Generative Pre-Training*（Radford et al., 2018），
用 **字符级自回归语言建模** 任务演示 GPT-1 的核心思想：**Decoder-only Transformer + 无监督预训练 + 有监督微调范式**。

本 Notebook 保留两条并行路径，并使用 **同一份小型语料、同一组提示词**：

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 看清 GPT-1 每个核心模块怎么计算 | 掌握更接近工程写法的 PyTorch 组装方式 |
| 实现方式 | **L1：从零手写** Decoder-only GPT-1 小模型 | **E2：PyTorch 内置模块**（`nn.TransformerEncoderLayer` + 因果掩码） |
| 代码量 | 更多，显式展示 attention / FFN / Post-LN / learned position embedding | 更少，把重复样板收进 `nn.TransformerEncoder` |
| 适合场景 | 面试准备、原理拆解、理解训练/推理差异 | 工程原型、快速验证、理解框架封装 |

> 两条路径不是两套无关代码，而是同一套 GPT-1 思想在不同抽象层级上的表达。


In [ ]:
import math
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")


## Section i：论文背景与任务定义

### 1. GPT-1 在历史上的位置
- GPT-1 的关键贡献不是“第一个 Transformer”，而是把 **生成式预训练 + 任务微调** 这条路线系统化。
- 预训练阶段优化语言模型目标：
  $$L_1(U) = \sum_i \log P(u_i \mid u_{i-k}, \dots, u_{i-1}; \Theta)$$
- 微调阶段在下游任务上再加任务头，并引入辅助语言模型损失：
  $$L_3(C) = L_2(C) + \lambda L_1(C)$$

### 2. 为什么这里用字符级语言建模演示 GPT-1
- 任务和论文的预训练目标一致：**根据前文预测下一个 token**。
- 字符级实现不代表 GPT-1 真实 tokenizer，而是为了在 CPU / 免费 Colab 上稳定跑通。
- 它足够小，却仍然保留了 GPT-1 最关键的结构与训练/推理差异。


### 3. 架构家族与本 Notebook 的边界

| 模型 | 结构 | 预训练目标 | 任务适配方式 |
|---|---|---|---|
| GPT-1 | Decoder-only Transformer | 自回归语言建模 | 预训练 + 微调 |
| GPT-2 | Decoder-only Transformer | 自回归语言建模 | Prompt / zero-shot 更突出 |
| BERT | Encoder-only Transformer | MLM + NSP | 预训练 + 微调 |

**本 Notebook 覆盖：**
- Decoder-only 架构
- learned token / position embedding
- masked multi-head self-attention
- FFN、残差、LayerNorm
- Teacher Forcing 训练
- 自回归推理
- PyTorch 内置模块的工程等价写法

**本 Notebook 不覆盖：**
- BookCorpus 级别的大规模预训练
- 原论文 12 层、1.17 亿参数规模
- 完整下游微调流水线的可运行复现


## Section ii：最小必要理论

### 1. GPT-1 的最小计算图
给定输入序列 `x_1, x_2, ..., x_T`，GPT-1 的计算可以压缩成 4 步：

1. **token embedding + learned position embedding**
2. **masked multi-head self-attention**
   $$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$
3. **position-wise FFN**
   $$\text{FFN}(x) = W_2\,\text{GELU}(W_1x + b_1) + b_2$$
4. **预测下一个 token**
   $$P(x_t \mid x_{<t}) = \text{softmax}(W_o h_t)$$

其中 `M` 是因果掩码：当前位置只能看见自己和过去，不能“偷看未来”。

### 2. 训练 vs 推理为什么不同
- **训练**：一次输入整段真实前文，配合 Teacher Forcing 并行预测每个位置的下一个 token。
- **推理**：从 prompt 开始，每次只新增 1 个 token，再把结果接回输入，形成自回归循环。

> 下面只保留理解代码所需的理论；更完整的背景请回看配套 `03_GPT1.md`。


### 组件映射表（Mandatory）

| 论文组件 | 学习路径覆盖 | 工程路径 / API 对应 | 状态 |
|---|---|---|---|
| Token Embedding | `nn.Embedding` 显式实现 | `nn.Embedding` | Runnable |
| Learned Positional Embedding | `nn.Embedding(seq_len, d_model)` 显式实现 | `nn.Embedding(seq_len, d_model)` | Runnable |
| Masked Multi-Head Self-Attention | 手写 `CausalSelfAttention` | `nn.TransformerEncoderLayer` 内部 attention + causal mask | Runnable |
| Position-wise FFN | 手写 `FeedForward` | `nn.TransformerEncoderLayer` 内部 FFN | Runnable |
| Residual + LayerNorm | 手写 Post-LN `GPT1Block` | `norm_first=False` 的 encoder layer | Runnable |
| Language Model Head | `lm_head` 输出词表 logits | `lm_head` 输出词表 logits | Runnable |
| Teacher Forcing 训练 | 共享 `train_language_model()` | 同一训练函数复用 | Runnable |
| Autoregressive 推理 | 共享 `generate_text()` | 同一推理函数复用 | Runnable |
| `[Extract]` 特殊 token | 仅在 markdown 解释其微调角色 | 不在本 notebook 中可运行复现 | Explain-only |
| `L_3 = L_2 + \lambda L_1` 微调目标 | 仅在 markdown 解释 | 不在本 notebook 中可运行复现 | Explain-only |


## Section 1：数据准备

这里不用真实的 BookCorpus，而用一组小型英文句子构成教学语料。这样做的目的不是复刻 GPT-1 的真实训练规模，而是让你能在 notebook 里：

- 直接看见 token 化后的序列
- 快速跑通损失下降
- 对比训练与推理的行为差异
- 在两条路径上使用 **同一任务、同一数据、同一 prompt**


In [ ]:
# ── Hyperparameters / 超参数集中管理 ──
SEQ_LEN = 64
BATCH_SIZE = 64
D_MODEL = 64
NUM_HEADS = 4
NUM_LAYERS = 2
D_FF = 128
DROPOUT = 0.1
LR = 3e-4
NUM_EPOCHS = 8
CORPUS_REPEAT = 10

raw_text = """
To be or not to be that is the question.
All that glitters is not gold.
A journey of a thousand miles begins with a single step.
The only thing we have to fear is fear itself.
In the middle of difficulty lies opportunity.
Life is what happens when you are busy making other plans.
The unexamined life is not worth living.
To infinity and beyond.
Knowledge is power.
I think therefore I am.
The pen is mightier than the sword.
Actions speak louder than words.
Fortune favors the bold.
Where there is a will there is a way.
Time flies over us but leaves its shadow behind.
The best way to predict the future is to create it.
Not all those who wander are lost.
Do or do not there is no try.
That which does not kill us makes us stronger.
The mind is everything what you think you become.
Happiness depends upon ourselves.
Stars cannot shine without darkness.
The only impossible journey is the one you never begin.
Dream big and dare to fail.
What we know is a drop what we do not know is an ocean.
It is during our darkest moments that we must focus to see the light.
The way to get started is to quit talking and begin doing.
You miss every shot you do not take.
Believe you can and you are halfway there.
In the end we only regret the chances we did not take.
""".strip()

raw_text = (raw_text + "\n") * CORPUS_REPEAT

print({
    "SEQ_LEN": SEQ_LEN,
    "BATCH_SIZE": BATCH_SIZE,
    "D_MODEL": D_MODEL,
    "NUM_HEADS": NUM_HEADS,
    "NUM_LAYERS": NUM_LAYERS,
    "D_FF": D_FF,
    "NUM_EPOCHS": NUM_EPOCHS,
    "CORPUS_CHARS": len(raw_text),
})


In [ ]:
chars = sorted(set(raw_text))
VOCAB_SIZE = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

def encode(text: str):
    return [char_to_idx[ch] for ch in text]

def decode(indices):
    return "".join(idx_to_char[i] for i in indices)

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Vocabulary preview: {''.join(chars)}")

sample = "The "
sample_ids = encode(sample)
print(f"Sample encode: {sample!r} -> {sample_ids}")
print(f"Sample decode: {decode(sample_ids)!r}")


In [ ]:
class CharDataset(Dataset):
    """Character-level next-token prediction dataset."""
    def __init__(self, text: str, seq_len: int):
        self.tokens = torch.tensor(encode(text), dtype=torch.long)      # (total_chars,)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.tokens) - self.seq_len

    def __getitem__(self, idx):
        x = self.tokens[idx: idx + self.seq_len]                        # (seq_len,)
        y = self.tokens[idx + 1: idx + self.seq_len + 1]               # (seq_len,)
        return x, y

dataset = CharDataset(raw_text, SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f"Dataset size: {len(dataset)}")
print(f"Batches per epoch: {len(dataloader)}")


In [ ]:
x_batch, y_batch = next(iter(dataloader))
print(f"Input batch shape:  {tuple(x_batch.shape)}")   # (batch, seq_len)
print(f"Target batch shape: {tuple(y_batch.shape)}")   # (batch, seq_len)
print(f"\nFirst input sample:\n{decode(x_batch[0].tolist())}")
print(f"\nFirst target sample:\n{decode(y_batch[0].tolist())}")
print("\nTeacher Forcing view: target is input shifted by one position.")


## Section 2：共享组件与训练工具

两条路径共享同一套任务接口：
- 输入：`(batch, seq_len)` 的整数 token 序列
- 输出：`(batch, seq_len, vocab_size)` 的 logits
- 训练：统一使用 next-token cross entropy
- 推理：统一使用自回归 `generate_text()`

这样可以把关注点放在“模块写法不同”，而不是“训练流程不同”。


In [ ]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def plot_loss_curve(loss_history, title: str):
    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(loss_history) + 1), loss_history, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def train_language_model(model: nn.Module, loader: DataLoader, epochs: int, lr: float, label: str):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for x, y in loader:
            x = x.to(device)                                              # (batch, seq_len)
            y = y.to(device)                                              # (batch, seq_len)

            logits = model(x)                                             # (batch, seq_len, vocab_size)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),                      # (batch * seq_len, vocab_size)
                y.reshape(-1),                                            # (batch * seq_len,)
            )

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        loss_history.append(avg_loss)

        if epoch == 1 or epoch % 2 == 0:
            print(f"[{label}] Epoch {epoch}/{epochs} - loss: {avg_loss:.4f}")

    return loss_history


@torch.no_grad()
def generate_text(model: nn.Module, prompt: str, max_new_tokens: int = 80,
                  temperature: float = 0.8, top_k: int = 0) -> str:
    model.eval()
    tokens = torch.tensor([encode(prompt)], dtype=torch.long, device=device)      # (1, prompt_len)

    for _ in range(max_new_tokens):
        tokens_cond = tokens[:, -SEQ_LEN:]                                         # (1, <= seq_len)
        logits = model(tokens_cond)[:, -1, :] / temperature                        # (1, vocab_size)

        if top_k > 0:
            k = min(top_k, logits.size(-1))
            values, _ = torch.topk(logits, k)
            logits = logits.masked_fill(logits < values[:, [-1]], float("-inf"))

        probs = F.softmax(logits, dim=-1)                                          # (1, vocab_size)
        next_token = torch.multinomial(probs, num_samples=1)                       # (1, 1)
        tokens = torch.cat([tokens, next_token], dim=1)                            # (1, current_len + 1)

    return decode(tokens[0].tolist())


## Section iii：学习路径（L1：从零手写 GPT-1）

### 组件 1：Masked Multi-Head Self-Attention

这是 GPT-1 的核心组件。它做两件事：
1. 把输入投影成 `Q / K / V`
2. 用因果掩码确保位置 `t` 只能看见 `<= t` 的 token

关键 shape：
- 输入：`(batch, seq, d_model)`
- 拆头后：`(batch, heads, seq, d_k)`
- 注意力分数：`(batch, heads, seq, seq)`
- 输出：`(batch, seq, d_model)`


In [ ]:
class CausalSelfAttention(nn.Module):
    """Hand-written masked multi-head self-attention for GPT-1."""
    def __init__(self, d_model: int, num_heads: int, seq_len: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()  # (seq, seq)
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x: torch.Tensor, return_attention: bool = False):
        batch_size, seq_len, d_model = x.shape                                      # (batch, seq, d_model)

        qkv = self.qkv_proj(x)                                                      # (batch, seq, 3 * d_model)
        q, k, v = qkv.chunk(3, dim=-1)                                              # each: (batch, seq, d_model)

        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        # q, k, v: (batch, heads, seq, head_dim)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)               # (batch, heads, seq, seq)
        scores = scores.masked_fill(self.causal_mask[:seq_len, :seq_len], float("-inf"))

        attn = F.softmax(scores, dim=-1)                                            # (batch, heads, seq, seq)
        attn = self.attn_dropout(attn)

        out = attn @ v                                                              # (batch, heads, seq, head_dim)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)   # (batch, seq, d_model)
        out = self.resid_dropout(self.out_proj(out))                                # (batch, seq, d_model)

        if return_attention:
            return out, attn
        return out


dummy_x = torch.randn(2, 12, D_MODEL)
dummy_out, dummy_attn = CausalSelfAttention(D_MODEL, NUM_HEADS, SEQ_LEN)(dummy_x, return_attention=True)
print(f"Attention output shape: {tuple(dummy_out.shape)}")
print(f"Attention matrix shape: {tuple(dummy_attn.shape)}")


### 组件 2：FFN、Residual、Post-LN Block 与完整 GPT-1

论文叙述下，GPT-1 更适合在教学里讲成 **Post-LN**：

$$h_1 = \text{LayerNorm}(x + \text{Attention}(x))$$
$$h_2 = \text{LayerNorm}(h_1 + \text{FFN}(h_1))$$

这和 GPT-2 常讲的 Pre-LN 有意区分开来。

同时，这里使用 **learned positional embedding**，而不是正弦位置编码，这也更贴近 GPT 系列的常见实现。


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)                                                         # (batch, seq, d_model)


class GPT1Block(nn.Module):
    """GPT-1 teaching block with Post-LN residual structure."""
    def __init__(self, d_model: int, num_heads: int, d_ff: int, seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.attn = CausalSelfAttention(d_model, num_heads, seq_len, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.ln1(x + self.attn(x))                                             # (batch, seq, d_model)
        x = self.ln2(x + self.ffn(x))                                              # (batch, seq, d_model)
        return x


class GPT1FromScratch(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 num_layers: int, d_ff: int, seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.seq_len = seq_len
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(seq_len, d_model)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            GPT1Block(d_model, num_heads, d_ff, seq_len, dropout)
            for _ in range(num_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(self._init_weights)
        self.lm_head.weight = self.token_embedding.weight                           # weight tying

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = idx.shape                                             # (batch, seq)
        positions = torch.arange(seq_len, device=idx.device).unsqueeze(0)          # (1, seq)

        x = self.token_embedding(idx) + self.position_embedding(positions)          # (batch, seq, d_model)
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x)                                                            # (batch, seq, d_model)

        logits = self.lm_head(x)                                                    # (batch, seq, vocab_size)
        return logits


learning_model = GPT1FromScratch(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    seq_len=SEQ_LEN,
    dropout=DROPOUT,
).to(device)

print(f"Learning-path parameter count: {count_parameters(learning_model):,}")
print(f"Forward output shape: {tuple(learning_model(x_batch[:2].to(device)).shape)}")


### 学习路径中的训练 vs 推理

**训练（Teacher Forcing）**
- 输入是完整真实前文
- 一个 forward 同时产生所有位置的 logits
- 目标是右移一位后的真实 token

**推理（Autoregressive generation）**
- 先给一个 prompt
- 每轮只取最后一个位置的 logits 采样 1 个 token
- 再把新 token 追加回输入，循环继续

这正是“同一模型、两种使用方式”的典型例子。


In [ ]:
print("=" * 70)
print("Learning path: train hand-written GPT-1")
print("=" * 70)
learning_loss = train_language_model(
    learning_model,
    dataloader,
    epochs=NUM_EPOCHS,
    lr=LR,
    label="Learning",
)
plot_loss_curve(learning_loss, "Learning Path Training Loss")


In [ ]:
shared_prompts = ["The ", "In the ", "To be "]
learning_generations = {}

print("=" * 70)
print("Learning path: autoregressive generation")
print("=" * 70)
for prompt in shared_prompts:
    text = generate_text(learning_model, prompt, max_new_tokens=80, temperature=0.7, top_k=5)
    learning_generations[prompt] = text
    print(f"\nPrompt: {prompt!r}")
    print(text)


## Section iv：工程路径（E2：PyTorch 内置模块的简洁实现）

这条路径不再手写 attention 和 block，而是把它们交给 PyTorch：

| 学习路径（手写） | 工程路径（内置 API） | 说明 |
|---|---|---|
| `CausalSelfAttention` | `nn.TransformerEncoderLayer` 内部 self-attention | 计算相同，抽象层更高 |
| `GPT1Block` | `nn.TransformerEncoderLayer(..., norm_first=False)` | 维持 Post-LN 讲法 |
| `ModuleList([...])` 堆叠 block | `nn.TransformerEncoder` | 少写很多样板代码 |
| 手动因果掩码逻辑 | `nn.Transformer.generate_square_subsequent_mask(T)` | 官方 API 直接生成 `(T, T)` mask |

PyTorch 官方文档语义下：
- `batch_first=True` 表示输入 shape 是 `(batch, seq, d_model)`
- `generate_square_subsequent_mask(T)` 返回自回归因果 mask

所以这条路径不是“另一个模型”，而是“同一个 GPT-1 思想的更简洁实现”。


In [ ]:
class GPT1Engineering(nn.Module):
    """Concise GPT-1 using PyTorch built-ins."""
    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 num_layers: int, d_ff: int, seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.seq_len = seq_len
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(seq_len, d_model)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,                                                # keep Post-LN viewpoint for GPT-1
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_layers,
            norm=None,
        )
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(self._init_weights)
        self.lm_head.weight = self.token_embedding.weight

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = idx.shape                                          # (batch, seq)
        positions = torch.arange(seq_len, device=idx.device).unsqueeze(0)       # (1, seq)

        x = self.token_embedding(idx) + self.position_embedding(positions)       # (batch, seq, d_model)
        x = self.dropout(x)

        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(idx.device)
        x = self.encoder(x, mask=causal_mask)                                    # (batch, seq, d_model)
        logits = self.lm_head(x)                                                 # (batch, seq, vocab_size)
        return logits


engineering_model = GPT1Engineering(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    seq_len=SEQ_LEN,
    dropout=DROPOUT,
).to(device)

print(f"Engineering-path parameter count: {count_parameters(engineering_model):,}")
print(f"Forward output shape: {tuple(engineering_model(x_batch[:2].to(device)).shape)}")


### 工程路径中的训练 vs 推理

**训练阶段**
- 训练目标和学习路径完全一致，仍然是 next-token prediction
- 差别只在于 block 内部细节被 `nn.TransformerEncoderLayer` 封装起来了

**推理阶段**
- 仍然是同一个 `generate_text()` 自回归循环
- 也仍然需要 causal mask，只是 forward 里由内置模块处理

一句话概括：**学习路径把算法摊开，工程路径把算法装进 API。**


In [ ]:
print("=" * 70)
print("Engineering path: train concise GPT-1")
print("=" * 70)
engineering_loss = train_language_model(
    engineering_model,
    dataloader,
    epochs=NUM_EPOCHS,
    lr=LR,
    label="Engineering",
)
plot_loss_curve(engineering_loss, "Engineering Path Training Loss")


In [ ]:
engineering_generations = {}

print("=" * 70)
print("Engineering path: autoregressive generation")
print("=" * 70)
for prompt in shared_prompts:
    text = generate_text(engineering_model, prompt, max_new_tokens=80, temperature=0.7, top_k=5)
    engineering_generations[prompt] = text
    print(f"\nPrompt: {prompt!r}")
    print(text)


## Section v：学习路径 vs 工程路径对比（Mandatory）

| 对比维度 | 学习路径 | 工程路径 |
|---|---|---|
| 教育价值 | 能直接看到 attention、FFN、residual、Post-LN、位置嵌入如何拼起来 | 能理解框架如何把同样的结构压缩成少量 API |
| 代码量 | 较多，适合逐层讲解 | 较少，适合快速复现与交付 |
| 灵活性 | 高：任何子模块都能单独改写 | 中：受限于 `nn.TransformerEncoderLayer` 的接口 |
| 稳定性 | 中：更容易因为实现细节出错 | 高：底层实现更成熟、样板更少 |
| 工业适配度 | 更适合教学、研究原型、debug 内部机制 | 更适合工程原型、快速验证、维护 |
| 适用场景 | 面试准备、算法理解、解释模型内部计算 | 项目落地、模型基线、快速实验 |


## Section vi：训练 vs 推理差异（两条路径都要说清楚）

| 阶段 | 学习路径行为 | 工程路径行为 |
|---|---|---|
| 训练 | `train_language_model()` 中一次输入完整序列，Teacher Forcing 并行预测每个位置 | 同一个训练函数，但 block 细节交给 `nn.TransformerEncoderLayer` |
| 掩码 | 手写 `causal_mask`，显式控制“不能看未来” | `generate_square_subsequent_mask(T)` 生成 `(T, T)` 因果 mask |
| 推理 | `generate_text()` 每轮追加 1 个 token，自回归展开 | 同一个 `generate_text()`，只是 forward 用的是内置模块 |
| 心智模型 | 算法完全展开 | 算法被封装，但本质不变 |

### 论文里的“预训练 vs 微调”差异
本 notebook 只把 **预训练目标** 做成可运行代码；论文中的微调部分（`[Extract]` token、`L_3 = L_2 + \lambda L_1`）保留为 explain-only 内容。

原因：
- 这份 notebook 的主线任务是 next-token prediction
- 如果再完整加入一个分类微调流水线，会把主线教学结构打散
- 但你仍然需要知道 GPT-1 的历史贡献并不只在语言建模，还在 **预训练 + 微调范式**


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(learning_loss) + 1), learning_loss, marker="o", label="Learning path")
plt.plot(range(1, len(engineering_loss) + 1), engineering_loss, marker="s", label="Engineering path")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning vs Engineering Training Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("=" * 90)
print(f"{'Prompt':<12s} | {'Learning path':<35s} | {'Engineering path':<35s}")
print("=" * 90)
for prompt in shared_prompts:
    left = learning_generations[prompt].replace("\n", " ")[:35]
    right = engineering_generations[prompt].replace("\n", " ")[:35]
    print(f"{prompt!r:<12s} | {left:<35s} | {right:<35s}")

print("\nParameter comparison")
print(f"Learning path params:    {count_parameters(learning_model):,}")
print(f"Engineering path params: {count_parameters(engineering_model):,}")
print(f"Final loss (learning):   {learning_loss[-1]:.4f}")
print(f"Final loss (engineering):{engineering_loss[-1]:.4f}")


## Section vii：结果、边界与何时使用哪条路径

### 结果应该怎么读
- 如果两条路径的 loss 都下降，说明它们都学到了同一个 next-token 任务。
- 如果生成文本开始出现训练语料的局部风格，说明 decoder-only 自回归机制已经工作。
- 不要把这份 notebook 的生成质量当成 GPT-1 原论文结果；这里的目标是 **教学可运行**，不是规模复现。

### 学习路径的边界
- **你得到的**：透明的模块级理解、对 shape 和 mask 的直觉、debug 能力。
- **你失去的**：实现更长、更容易写错、训练稳定性更依赖细节。

### 工程路径的边界
- **你得到的**：更短的代码、更成熟的 API、更容易复用。
- **你失去的**：内部细节被隐藏，不利于第一次建立完整心智模型。

### 实践建议
- 第一次学 GPT-1：先走学习路径。
- 要在项目里快速搭基线：走工程路径。
- 要做面试表达：一定要能来回映射“手写 attention”与“内置 encoder layer”。


## Appendix A：Attention 可视化（optional）

下面把学习路径里一个样本的 attention heatmap 画出来。重点不是数值本身，而是观察：
- 注意力矩阵是下三角可见的
- 未来位置被掩码掉
- 每一行代表当前位置如何汇聚历史信息


In [ ]:
viz_tokens = x_batch[:1].to(device)                                                   # (1, seq_len)
viz_positions = torch.arange(viz_tokens.size(1), device=device).unsqueeze(0)           # (1, seq_len)
viz_hidden = learning_model.token_embedding(viz_tokens) + learning_model.position_embedding(viz_positions)
_, viz_attn = learning_model.blocks[0].attn(viz_hidden, return_attention=True)

head0 = viz_attn[0, 0].detach().cpu().numpy()                                           # (seq, seq)
plt.figure(figsize=(6, 5))
plt.imshow(head0[:20, :20], cmap="magma")
plt.colorbar()
plt.title("Causal Attention Heatmap (Head 0)")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.tight_layout()
plt.show()


## Section viii：面试与项目表达（Mandatory）

### 高频面试题

**Q1：GPT-1 的核心贡献是什么？**
- 不是单纯“做了一个大点的 Transformer”。
- 它第一次系统证明：**无监督预训练得到的语言表示，可以迁移到多种下游任务**。
- 真正改变行业的是“预训练 + 微调”这条范式。

**Q2：为什么 GPT-1 适合生成任务？**
- 因为它是 **Decoder-only** 架构。
- 每个位置只能看过去，天然符合“根据前文生成后文”的因果方向。
- 这和 BERT 的双向编码器目标不同。

**Q3：Teacher Forcing 和自回归推理有什么区别？**
- 训练时使用真实历史 token，能并行算完整序列，效率高。
- 推理时没有真实未来，只能一步一步往后生成。
- 所以同一个模型，训练和推理的调用方式明显不同。

**Q4：为什么必须要 causal mask？**
- 没有 causal mask，模型训练时会直接看到未来 token。
- 那样 loss 会虚高地变好，但推理时无法复现同样条件。
- 所以 causal mask 是 decoder-only 语言模型的基本约束。

**Q5：GPT-1 和 GPT-2 的一个关键结构区别是什么？**
- GPT-1 常被讲为 Post-LN，GPT-2 更典型地强调 Pre-LN 与更大规模训练。
- GPT-2 还更突出 Prompt / zero-shot 使用方式。
- 面试里最好把“结构差异”和“范式差异”分开说。


### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|---|---|
| 1 | GPT-1 本质在做什么？ | 用 decoder-only Transformer 做自回归语言建模，并把预训练表示迁移到下游任务。 |
| 2 | 为什么要 causal mask？ | 防止训练时偷看未来，保证训练条件和推理条件一致。 |
| 3 | Teacher Forcing 的价值是什么？ | 让 next-token prediction 在训练时可以并行计算整段序列。 |
| 4 | `[Extract]` token 是干什么的？ | 在微调任务里充当序列级摘要位置，类似把上下文压成一个可分类向量。 |
| 5 | `L_3 = L_2 + \lambda L_1` 说明了什么？ | GPT-1 微调时保留语言模型目标，避免灾难性遗忘并帮助泛化。 |
| 6 | 学习路径和工程路径如何对应？ | 手写 block 对应内置 encoder layer；手写 mask 对应官方 causal mask API。 |
| 7 | GPT-1 和 BERT 的根本差别？ | GPT-1 是单向生成式预训练，BERT 是双向编码式预训练。 |

### 项目表达

> 如果面试官问“你做过什么相关项目”，可以这样组织：

- **原理理解**：我手写过一个教学版 GPT-1，能解释 attention、FFN、LayerNorm、位置嵌入和自回归生成。
- **工程能力**：我也用 PyTorch 内置模块重写过同一个模型，知道怎么把原理映射到工程 API。
- **范式理解**：我能讲清 GPT-1 真正重要的是“预训练 + 微调”，而不只是一个 decoder-only 模型。


### 延伸阅读与对比

| 对比维度 | GPT-1 | GPT-2 | BERT |
|---|---|---|---|
| 结构 | Decoder-only | Decoder-only | Encoder-only |
| 预训练目标 | 自回归 LM | 自回归 LM | MLM (+ NSP in original BERT) |
| 代表范式 | 预训练 + 微调 | 更强的 Prompt / zero-shot | 编码表示迁移 |
| 优势 | 奠定 GPT 路线 | 更强生成与零样本能力 | 更强双向理解 |
| 典型问题 | 需要逐 token 推理 | 仍然自回归、推理慢 | 不擅长原生生成 |

### 进阶探索方向
- 把字符级 notebook 升级为子词级 tokenizer 版本。
- 把单纯预训练演示升级为一个轻量级分类微调演示。
- 比较 Post-LN GPT-1 与 Pre-LN GPT-2 在训练稳定性上的差异。
- 进一步解释 KV-cache 为什么能加速大模型推理。
